# Lesson 4: Bring an existing Brain-Score model

**You will learn:** how models registered for classic Brain-Score become subjects, and how to translate one into a `BrainScoreModel`.
**Requires:** CPU only, under a minute.

Classic vision models are `BrainModel` objects with methods such as `look_at` and `start_recording`; language models are `ArtificialSubject` objects with `digest_text`. You do not rewrite them: `brainscore.load_model(...)` wraps them in an **adapter** that presents them as a subject, and classic benchmarks still run on any subject.

**The check:** does the classic registration and its translation record exactly the same activity from the same network?

## Setup

Imports and settings.

In [1]:
# Keep the output short: no progress bars, no library warnings.
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
import tqdm as _tqdm
for _bar in (tqdm, _tqdm.tqdm):
    _bar.__init__ = partialmethod(_bar.__init__, disable=True)
warnings.filterwarnings('ignore')
import os
os.environ['BS_INSTALL_DEPENDENCIES'] = 'no'      # every requirement is already installed

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image

ASSETS = Path('../assets/lessons')        # weights and images committed with the lessons

# SqueezeNet 1.1 trained on ImageNet (5 MB). These images come from another dataset, CIFAR-10.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()                                # inference only: no dropout, no training

# The network expects 224 x 224 images, normalized the way it was trained.
TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files -> one (batch, 3, 224, 224) array."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## The subject

### A classic vision registration

This is how a model is registered for classic Brain-Score: a `PytorchWrapper` records layers, and a `ModelCommitment` says which layer stands for which region.

In [2]:
from brainscore_vision.model_helpers.activations.pytorch import PytorchWrapper
from brainscore_vision.model_helpers.brain_transformation import ModelCommitment

legacy_model = ModelCommitment(
    identifier='squeezenet-classic',
    activations_model=PytorchWrapper(identifier='squeezenet-classic', model=net, preprocessing=preprocess),
    layers=['features.12'],
    region_layer_map={'IT': 'features.12'},
)

### The adapter makes it a subject

In [3]:
from brainscore_vision.compat.unified_adapter import VisionModelAdapter

classic = VisionModelAdapter(legacy_model)
print('outputs:', classic.out_channels)

outputs: {'neural:IT', 'behavior'}


### Translating the registration

| Classic | `BrainScoreModel` |
| --- | --- |
| `PytorchWrapper(..., preprocessing=...)` | `VisionWrapper(..., preprocessing=...)` in `extractor={'vision': ...}` |
| `region_layer_map={'IT': ...}` | `region_layer_map={'IT': ...}` |
| `look_at(stimuli)` | `participate(session)` (or `process(stimuli)` interactively) |

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

translated = BrainScoreModel(
    identifier='squeezenet-new', model=net,
    extractor={'vision': VisionWrapper(identifier='squeezenet-new', model=net, preprocessing=preprocess)},
    region_layer_map={'IT': 'features.12'})

## The experiment

### Stimuli

In [5]:
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images, 12 of each CIFAR-10 category, each labelled animal or vehicle.
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

### Task

Plain viewing: each image is shown once and IT is recorded. The classic model also needs a recording window, `time_bins=[(70, 170)]` (milliseconds after image onset), as classic benchmarks give it.

## Run

The same images go to both subjects, each in its own session.

In [6]:
from brainscore_core.streaming import BufferedSession

session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'], time_bins=[(70, 170)])
classic.participate(session)
legacy_activity = session.collect()

session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:IT'])
translated.participate(session)
new_activity = session.collect()

## Read results back

Both hold one row per image and one column per unit of `features.12`.

In [7]:
print('classic:', legacy_activity.dims, legacy_activity.shape)
print('translated:', new_activity.dims, new_activity.shape)

classic: ('presentation', 'neuroid') (120, 86528)
translated: ('presentation', 'neuroid') (120, 86528)


## Analysis

The two assemblies may list units in a different order, so the comparison sorts all values first.

In [8]:
def same_values(a, b):
    """True when two arrays hold the same numbers, in any order."""
    return np.allclose(np.sort(a, axis=None), np.sort(b, axis=None))

## Results

In [9]:
print('same activity:', same_values(np.asarray(new_activity.values), np.asarray(legacy_activity.values)))

same activity: True


The two record the same numbers from the same network: translating changes the interface, not the measurement.

Registered models need no wrapping: `brainscore.load_model('alexnet')` returns a subject, classic or new, and `brainscore.score(model, benchmark)` runs any benchmark on it.

## Summary

- Adapters make classic models subjects; classic benchmarks run on any subject.
- A translation is one to one, and it records the same activity.

**Next:** [Lesson 5](05_layers_and_regions.ipynb) compares layers against a null network.